# 03 — DecodabilityStage 2. An embedding is not evidence. Here we turn structure into a number a downstream reader could actually use.Every accuracy in this notebook comes with a control. That is not politeness; an accuracy without a chance level and a held-out split does not mean anything.

In [ ]:
%load_ext autoreload%autoreload 2import numpy as npimport matplotlib.pyplot as pltimport pandas as pdfrom compbio2026 import data, decoding, geometry, plottingplotting.apply_style()rng = np.random.default_rng(2026)shd = data.load("train")idx = np.flatnonzero(shd.english_mask())idx = rng.choice(idx, size=min(2500, len(idx)), replace=False)X, y, t = data.build_design_matrix(shd, bin_ms=10.0, t_max_ms=800.0, smooth_ms=10.0, trials=idx)Xf = data.flatten(X)speaker = shd.speaker[idx]print(Xf.shape, "chance =", round(decoding.chance_level(y), 3))

## A first number, with its controls

In [ ]:
res = decoding.decode(Xf, y, estimator=decoding.make_readout("ridge"))null = decoding.shuffle_control(Xf, y, n_repeats=5)print(f"accuracy      {res['mean']:.3f} +/- {res['std']:.3f}")print(f"chance        {res['chance']:.3f}")print(f"shuffled      {null.mean():.3f} +/- {null.std():.3f}")

The shuffled-label control is a stronger null than chance: it absorbs any leakage the pipeline happens to have. If your real accuracy is not clearly above that distribution, you have not shown anything.

## Random split vs. speaker held outRandom cross-validation lets the classifier see other utterances by the same speaker. Holding whole speakers out asks whether you learned the digit or the voice. **The gap between these two numbers is worth reporting on its own.**

In [ ]:
res_rand = decoding.decode(Xf, y)res_spk = decoding.decode(Xf, y, groups=speaker)print(f"random split       {res_rand['mean']:.3f}")print(f"speaker held out   {res_spk['mean']:.3f}")print(f"generalisation gap {res_rand['mean'] - res_spk['mean']:.3f}")

## Reproducing the published baselineCramer et al. 2020, Figure 3: classifiers with no access to spike timing plateau near **60 %**; temporally aware ones reach **~85 %**.Collapsing time (one bin for the whole trial) is the "no timing" condition. Keeping bins is the "with timing" condition. You will not match their numbers exactly — different classifier, different subset — but the *gap* should reproduce, and if it does not, something in your pipeline is wrong.

In [ ]:
X_counts = X.sum(axis=2)                       # (trials, channels) - timing destroyedres_counts = decoding.decode(X_counts, y)res_timed = decoding.decode(Xf, y)print(f"spike counts only   {res_counts['mean']:.3f}")print(f"with spike timing   {res_timed['mean']:.3f}")

## Learning curvesAre you data-limited or representation-limited? A curve that has flattened means more data will not help and the ceiling is in the representation itself.

In [ ]:
n_tr, m_tr, s_tr = decoding.learning_curve_trials(Xf, y)n_ch, m_ch, s_ch = decoding.learning_curve_channels(X, y, n_repeats=3)fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))plotting.learning_curve(n_tr, m_tr, s_tr, ax=axes[0], chance=res["chance"], xlabel="Training trials")plotting.learning_curve(n_ch, m_ch, s_ch, ax=axes[1], chance=res["chance"], xlabel="Channels (random subsets)")axes[1].set_xscale("log")fig.tight_layout()

The channel curve is the important one for this project. If accuracy is near-maximal with 50 of 700 channels, **the code is massively redundant** — and that is the fact Stage 3 has to reckon with. It also means "these channels are important" will be very hard to establish.

## When does the digit become decodable?With `cumulative=True` the readout at time *t* sees every bin up to *t*, which is what a downstream reader would have.

In [ ]:
t_ms, acc = decoding.accuracy_over_time(X, y, t, cumulative=True)t_ms_i, acc_i = decoding.accuracy_over_time(X, y, t, cumulative=False)fig, ax = plt.subplots(figsize=(6, 3.6))ax.plot(t_ms, acc, label="cumulative", color=plotting.ACCENT)ax.plot(t_ms_i, acc_i, label="single bin", color=plotting.PALETTE[1])ax.axhline(res["chance"], ls="--", lw=1, color=plotting.INK_MUTED)ax.set_xlabel("Time from trial onset (ms)")ax.set_ylabel("Accuracy")ax.legend()

## The actual question of this project**Does any geometric property predict decodability?** This is assumed constantly in the population-coding literature and tested rarely.Sweep preprocessing conditions, compute geometry and accuracy for each, and regress.

In [ ]:
rows = []for bin_ms in [5.0, 10.0, 20.0, 50.0]:    for smooth_ms in [0.0, 10.0, 30.0]:        Xc, yc, _ = data.build_design_matrix(shd, bin_ms=bin_ms, t_max_ms=800.0,                                             smooth_ms=smooth_ms, trials=idx)        Xcf = data.flatten(Xc)        g = geometry.summarize(Xcf, yc)        d = decoding.decode(Xcf, yc, estimator=decoding.make_readout("ridge"))        rows.append({"bin_ms": bin_ms, "smooth_ms": smooth_ms, "accuracy": d["mean"], **g})sweep = pd.DataFrame(rows)sweep.to_csv("../data/geometry_vs_decoding.csv", index=False)sweep.round(3)

In [ ]:
props = ["participation_ratio", "class_separation", "between_class_distance", "within_class_radius"]fig, axes = plt.subplots(1, len(props), figsize=(4 * len(props), 3.3))for ax, p in zip(axes, props):    ax.scatter(sweep[p], sweep["accuracy"], color=plotting.ACCENT, s=28, linewidths=0)    r = np.corrcoef(sweep[p], sweep["accuracy"])[0, 1]    ax.set_xlabel(p.replace("_", " "))    ax.set_title(f"r = {r:.2f}")axes[0].set_ylabel("Decoding accuracy")fig.tight_layout()

**Interpret this carefully.** A correlation across preprocessing conditions is not the same as a causal claim about geometry. Both quantities are functions of the same choices, so they can co-vary without one predicting the other in any useful sense. Say so, and think about what a better test would look like.---## Exercises1. **Ridge vs. logistic.** Do they agree? If not, which do you trust and why?2. **Confusion structure.** Plot the confusion matrix. Which digits are confused? Does the pattern make acoustic sense?3. **A non-linear ceiling.** Fit a small MLP or an RBF-kernel SVM. How much is left on the table by a linear readout? That gap bounds how much of the structure is linearly accessible.4. **The right null for timing.** Compare against your temporal-shuffle surrogate from notebook 01 rather than against summed counts. Does the "timing matters" conclusion survive the stricter control?